In [2]:
import os
import pandas as pd
import statsmodels.api as sm
import numpy as np

In [3]:
print(os.getcwd())

/Users/bettyk/Desktop/Risk_Factor_Analysis_of_Obstructive-Sleep_Apnea/10_22_Betty


In [4]:
os.listdir()

['age_over_80_logistic_regression.csv',
 'male_logistic_regression.csv',
 'Female_logistic_regression.ipynb',
 'Total_logistic_regression.ipynb',
 'age_over_80_logistic_regression.ipynb',
 '.ipynb_checkpoints',
 'Male_logistic_regression.ipynb',
 'Table2_logistic_regression.csv',
 'total_logistic_regression.csv']

In [5]:
# Step 1. 載入基本套件

# 讀取資料
df = pd.read_csv("../raw_data/female.csv")

# 查看前幾列資料，確定有成功載入
print(df.head())
print()

# 檢查欄位名稱、資料型態、缺漏值
print(df.info())
print()
print(df.isnull().sum())

   Second_Stroke  age  sex  HLOS  NIHSS  tPA(0/1)  EVT(0/1)  HTN(0/1)  \
0              0   71    1    12      5         0         0         0   
1              0   65    1    14      9         0         0         0   
2              0   62    1    13      2         0         0         1   
3              0   91    1    18     32         0         0         1   
4              0   77    1    10      0         0         0         1   

   DM(0/1)  Dyslipidemia(0/1)  Af(0/1)  smoking(Y/N/Q)   LDL   cholesterol  \
0        0                  0        0               0   81.7          180   
1        0                  1        0               0  116.5          235   
2        0                  1        0               0  116.3          187   
3        1                  0        0               0   78.2          127   
4        1                  1        0               0  105.8          177   

    TG   Cre  SGPT  HbA1c  MRS  
0   57  0.35  80.0    5.1    4  
1  155  0.37  87.0    4.6 

In [6]:
# 清理欄位名稱（避免空白、括號、特殊符號）
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
df.columns = df.columns.str.replace(r'_+$', '', regex=True)

# 把 N/Q 都合併成 0（非吸菸），Y=1
# 假設 smoking 欄位是 0=不吸菸, 1=吸菸, 2=戒菸
if 'smoking_Y_N_Q_' in df.columns:
    df['smoking_Y_N_Q_'] = df['smoking_Y_N_Q_'].replace({2: 0})
# print(df['smoking_Y_N_Q_'].value_counts())

# 目標變數 (Y)
Y = df['Second_Stroke']

# 自變數 (X) - 把你要放進模型的欄位都列出來
X = df.drop(columns=['Second_Stroke','sex'])

# 只保留數值型欄位（防止類別文字造成報錯）
X = X.select_dtypes(include=[np.number])

# 移除缺值（Y + X 一起）
d = pd.concat([Y, X], axis=1).dropna()
Y = d['Second_Stroke']
X = d.drop(columns=['Second_Stroke'])

# 增加截距項 (Intercept)
X = sm.add_constant(X)

# 建立 Logistic Regression 模型
logit_model = sm.Logit(Y, X)
result = logit_model.fit(disp=False)

# 輸出結果摘要
print(result.summary())

                           Logit Regression Results                           
Dep. Variable:          Second_Stroke   No. Observations:                  568
Model:                          Logit   Df Residuals:                      550
Method:                           MLE   Df Model:                           17
Date:                Mon, 06 Oct 2025   Pseudo R-squ.:                 0.07867
Time:                        20:56:51   Log-Likelihood:                -94.458
converged:                       True   LL-Null:                       -102.52
Covariance Type:            nonrobust   LLR p-value:                    0.5146
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -0.7001      1.976     -0.354      0.723      -4.573       3.173
age                  0.0092      0.020      0.454      0.650      -0.031       0.049
HLOS              5.661e-05 

In [7]:
# print(result.params)

In [8]:
# print(result.pvalues)

In [9]:
# print(result.conf_int())

In [10]:
# ==== 3. 整理成 Table（Adjusted OR, 95% CI, P value） ====
# 丟掉截距
params = result.params.drop(labels=["const"], errors="ignore")
pvals  = result.pvalues.drop(labels=["const"], errors="ignore")
conf   = result.conf_int().rename(columns={0:"lower",1:"upper"}).drop(labels=["const"], errors="ignore")

In [11]:
# print(result.params)

In [12]:
# 轉成 OR 與 95% CI (OR)
OR   = np.exp(params)
CI_l = np.exp(conf["lower"])
CI_u = np.exp(conf["upper"])

In [13]:
# 排序照原本欄位順序
order = [c for c in d.columns if c != "Second_Stroke" and c in params.index]
table = pd.DataFrame({
    "Adjusted odd's ratio": OR.reindex(order),
    "95 % Confidence Interval": [f"{CI_l.reindex(order)[i]:.2f}-{CI_u.reindex(order)[i]:.2f}" for i in order],
    "P value": [("<0.01" if pvals.reindex(order)[i] < 0.01 else f"{pvals.reindex(order)[i]:.2f}") for i in order]
}, index=order)

In [14]:
# 顯著性星號（p<0.05）
stars = pvals.reindex(order).apply(lambda p: "*" if p < 0.05 else "")
table["P value"] = table["P value"] + stars

In [15]:
# OR 四捨五入到兩位（看你期刊格式調整）
table["Adjusted odd's ratio"] = table["Adjusted odd's ratio"].round(2)

In [16]:
# ==== 4.（可選）把變數名稱換成好看的顯示文字 ====

# # 建立一個 mapping dictionary：左邊是原始欄位名，右邊是報表要顯示的名稱
# pretty_names = {
#     'age': 'Age (years)',
#     'HLOS': 'Hospital length of stay (days)',
#     'NIHSS': 'NIH Stroke Scale score',
#     'tPA_0_1_': 'tPA (tissue Plasminogen Activator) treatment (0=No, 1=Yes)',
#     'EVT_0_1_': 'EVT (Endovascular thrombectomy)(0/1)',
#     'HTN_0_1_': 'Hypertension (0/1)',
#     'DM_0_1_': 'Diabetes mellitus (0/1)',
#     'Dyslipidemia_0_1_': 'Dyslipidemia (0/1)',
#     'Af_0_1_': 'Atrial fibrillation (0/1)',
#     'smoking_Y_N_Q_': 'Smoking (Y/N/Q)',
#     'LDL': 'LDL (mg/dL)',
#     'cholesterol': 'Total cholesterol (mg/dL)',
#     'TG': 'Triglycerides (mg/dL)',
#     'Cre': 'Creatinine (mg/dL)',
#     'SGPT': 'ALT (SGPT, U/L)',
#     'HbA1c': 'HbA1c (%)',
#     'MRS': 'Modified Rankin Scale at discharge'
# }

# # 使用 pandas 的 rename()，將索引(index) 改成好看的名字
# table = table.rename(index=pretty_names)

# 顯示表格
print("\n=== Multivariate logistic regression (Adjusted) ===")
display(table)

# （可選）存成 CSV，直接貼到論文表格
table.to_csv("female_logistic_regression.csv", encoding="utf-8-sig")


=== Multivariate logistic regression (Adjusted) ===


,Adjusted odd's ratio,95 % Confidence Interval,P value
age,1.01,0.97-1.05,0.65
HLOS,1.00,0.96-1.04,1.00
NIHSS,1.00,0.92-1.08,0.95
tPA_0_1,0.39,0.04-3.44,0.40
EVT_0_1,1.26,0.19-8.60,0.81
HTN_0_1,0.54,0.21-1.41,0.21
DM_0_1,0.98,0.32-3.01,0.97
Dyslipidemia_0_1,1.27,0.47-3.42,0.64
Af_0_1,1.24,0.40-3.87,0.71
smoking_Y_N_Q,0.69,0.08-6.25,0.74
